In [ ]:
!kaggle datasets download -d ayushmandatta1/deepdetect-2025
!unzip deepdetect-2025.zip -d deepfake_dataset


Streaming output truncated to the last 5000 lines.
  inflating: deepfake_dataset/ddata/train/real/61451.jpg  
  inflating: deepfake_dataset/ddata/train/real/61453.jpg  
  inflating: deepfake_dataset/ddata/train/real/61455.jpg  
  inflating: deepfake_dataset/ddata/train/real/61456.jpg  
  inflating: deepfake_dataset/ddata/train/real/61458.jpg  
  inflating: deepfake_dataset/ddata/train/real/61459.jpg  
  inflating: deepfake_dataset/ddata/train/real/61460.jpg  
  inflating: deepfake_dataset/ddata/train/real/61461.jpg  
  inflating: deepfake_dataset/ddata/train/real/61464.jpg  
  inflating: deepfake_dataset/ddata/train/real/61465.jpg  
  inflating: deepfake_dataset/ddata/train/real/61466.jpg  
  inflating: deepfake_dataset/ddata/train/real/61467.jpg  
  inflating: deepfake_dataset/ddata/train/real/61468.jpg  
  inflating: deepfake_dataset/ddata/train/real/61469.jpg  
  inflating: deepfake_dataset/ddata/train/real/61470.jpg  
  inflating: deepfake_dataset/ddata/train/real/61472.jpg  
  inf

In [ ]:
import os
os.listdir("deepfake_dataset")


['ddata']

In [ ]:
os.listdir("deepfake_dataset/ddata")


['train', 'test']

In [ ]:
os.listdir("deepfake_dataset/ddata/train")


['fake', 'real']

In [ ]:
os.listdir("deepfake_dataset/ddata/test")


['fake', 'real']

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!rm -rf /content/drive/MyDrive/deepfake_dataset


In [ ]:
!mv deepfake_dataset /content/drive/MyDrive/


In [ ]:
os.listdir('/content/drive/MyDrive/deepfake_dataset/ddata')


['test', 'train']

In [ ]:
BASE_DIR = "/content/drive/MyDrive/deepfake_dataset/ddata"

TRAIN_DIR = BASE_DIR + "/train"
TEST_DIR  = BASE_DIR + "/test"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "/content/drive/MyDrive/deepfake_dataset/ddata"

TRAIN_DIR = BASE_DIR + "/train"
TEST_DIR  = BASE_DIR + "/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)


Found 90409 images belonging to 2 classes.
Found 21776 images belonging to 2 classes.


In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
for layer in base_model.layers:
    layer.trainable = False


In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history = model.fit(
    train_generator,
    epochs=5,
    validation_data=test_generator
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1815s 637ms/step - accuracy: 0.5873 - loss: 0.6710 - val_accuracy: 0.8120 - val_loss: 0.5080
Epoch 2/5
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1711s 605ms/step - accuracy: 0.6613 - loss: 0.6198 - val_accuracy: 0.8078 - val_loss: 0.4786
Epoch 3/5
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1756s 621ms/step - accuracy: 0.6839 - loss: 0.5984 - val_accuracy: 0.8087 - val_loss: 0.4614
Epoch 4/5
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1711s 605ms/step - accuracy: 0.6972 - loss: 0.5822 - val_accuracy: 0.8354 - val_loss: 0.4333
Epoch 5/5
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1734s 614ms/step - accuracy: 0.7044 - loss: 0.5738 - val_accuracy: 0.8237 - val_loss: 0.4352


In [ ]:
loss, acc = model.evaluate(test_generator)
print("VGG16 Accuracy:", acc)


681/681 ━━━━━━━━━━━━━━━━━━━━ 140s 205ms/step - accuracy: 0.7677 - loss: 0.5033
VGG16 Accuracy: 0.8236590623855591


In [ ]:
model.save("/content/drive/MyDrive/deepfake_vgg16.h5")
